# SEACO-6161 · 01a DBFS scan · `dbw-arc-merone-pipeline-prod`

**Read-only.** This notebook measures how much of `dbfs:/` each directory holds (goal 1 of the ticket), on an
account of 92.45 TiB. Measuring means listing every file, so it is built to finish in bounded time:

| Problem with a large DBFS | How this notebook handles it |
|---|---|
| Tens of thousands of directories to size | Sizing runs on the **executors**, in Scala, one Spark task per group of directories. More workers make it faster |
| Walking the tree is slow too | The plan is built level by level, with parallel listing calls |
| A run may not finish in one go | Directories are sized in **batches**, and every batch is saved at once. The run stops when `time_budget_min` is used up |
| Cancelling used to lose everything | Set `resume_run_id` to an earlier run id and it continues where that run stopped. The plan is saved too, so it is not rebuilt |

**Cluster:** Single User access mode. Throughput is set by the total executor cores, so for the first full scan
use 8–16 workers of a general-purpose size such as `Standard_D8ds_v5`. You can run it as a one-off job.
It can run at the same time as `01_investigation` on another cluster.

**Depth:** `scan_depth = 3` gives the breakdown down to, for example, `/user/hive/warehouse` or
`/mnt/apt/warehouse`. A deeper scan gives finer detail but more directories. The total size is the same either way.

In [0]:
%run ./00_config_and_helpers

In [0]:
dbutils.widgets.text("scan_depth", "3", "1. Depth (levels below dbfs:/)")
dbutils.widgets.text("max_children", "1000", "2. Stop descending below dirs with more children")
dbutils.widgets.text("batch_size", "2000", "3. Directories per saved batch")
dbutils.widgets.text("time_budget_min", "60", "4. Stop after N minutes (resume later)")
dbutils.widgets.text("resume_run_id", "", "5. Resume this run id (empty = new scan)")
dbutils.widgets.dropdown("scan_external_mounts", "false", ["true", "false"], "6. Also size external mounts")

SCAN_DEPTH = int(dbutils.widgets.get("scan_depth"))
MAX_CHILDREN = int(dbutils.widgets.get("max_children"))
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
TIME_BUDGET_MIN = float(dbutils.widgets.get("time_budget_min"))
RESUME_RUN_ID = dbutils.widgets.get("resume_run_id").strip()
SCAN_EXTERNAL = dbutils.widgets.get("scan_external_mounts") == "true"

## 1 · Plan (or reload the plan of the run being resumed)

`dbfs:/` is split into leaf directories that each sit on one storage account. A directory with a mount point
below it is always opened up, so no bytes get counted under the wrong account. When a directory has more than
`max_children` entries, its children become leaves and the walk stops there. This keeps the plan cheap and
the batches balanced. `/databricks-datasets` and UC Volumes are skipped.

In [0]:
SKIP_PREFIXES = ("/databricks-datasets", "/Volumes", "/volumes", "/volume")
PLAN_SCHEMA = "run_id string, run_ts timestamp, workspace string, path string, storage_class string, kind string, bytes long"
SCAN_SCHEMA = ("run_id string, run_ts timestamp, workspace string, path string, storage_class string, "
               "kind string, bytes long, files long, dirs long, error string, millis long")


def plan_leaves(max_depth, max_children):
    """Breadth-first walk, one level at a time, with parallel ls calls. Returns (leaf_dirs, loose_files)."""
    leaves, files, frontier, depth = [], [], ["/"], 0
    with ThreadPoolExecutor(THREADS * 2) as pool:
        while frontier:
            listings = list(pool.map(lambda p: (p, ls("dbfs:" + p)), frontier))
            nxt = []
            for parent, children in listings:
                wide = len(children) > max_children
                for c in children:
                    cp = normalize(c.path)
                    if not c.isDir():
                        files.append((cp, int(c.size)))
                        continue
                    if cp.startswith(SKIP_PREFIXES):
                        continue
                    nested_mount = any(mp.startswith(cp + "/") for mp in MOUNT_POINTS)
                    if nested_mount or (depth + 1 < max_depth and not wide):
                        nxt.append(cp)
                    else:
                        leaves.append(cp)
            depth += 1
            print(f"  level {depth}: listed {len(frontier):,} dirs -> {len(leaves):,} leaves so far, "
                  f"{len(nxt):,} to open next")
            frontier = nxt
    return leaves, files


t0 = time.time()
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    plan_df = spark.table(audit_table("dbfs_scan_plan")).filter(F.col("run_id") == RUN_ID)
    RUN_TS = plan_df.select("run_ts").first()[0]
else:
    leaves, loose = plan_leaves(SCAN_DEPTH, MAX_CHILDREN)
    plan_rows = [dict(path=p, storage_class=classify_path(p), kind="dir", bytes=None) for p in leaves] + \
                [dict(path=p, storage_class=classify_path(p), kind="file", bytes=b) for p, b in loose]
    save_audit("dbfs_scan_plan", plan_rows, schema=PLAN_SCHEMA)
    # loose files already have their size: store them as scan results straight away
    save_audit("dbfs_scan", [dict(path=r["path"], storage_class=r["storage_class"], kind="file", bytes=r["bytes"],
                                  files=1, dirs=0, error=None, millis=0) for r in plan_rows if r["kind"] == "file"],
               schema=SCAN_SCHEMA)
    plan_df = spark.table(audit_table("dbfs_scan_plan")).filter(F.col("run_id") == RUN_ID)
PLAN_SECONDS = time.time() - t0

wanted = plan_df.filter("kind = 'dir'")
if not SCAN_EXTERNAL:
    wanted = wanted.filter("storage_class = 'root'")
try:
    already = spark.table(audit_table("dbfs_scan")).filter((F.col("run_id") == RUN_ID) & (F.col("kind") == "dir")) \
        .select("path").distinct()
    remaining = wanted.join(already, "path", "left_anti")
except Exception:
    remaining = wanted
remaining.select("path", "storage_class").cache().createOrReplaceTempView("_scan_remaining")
PLAN_COUNTS = dict(planned=wanted.count(), remaining=spark.table("_scan_remaining").count())

In [0]:
print(f"run_id {RUN_ID} ({'resumed' if RESUME_RUN_ID else 'new'}), plan ready in {PLAN_SECONDS:,.0f}s")
print(f"directories to size: {PLAN_COUNTS['planned']:,} planned, {PLAN_COUNTS['remaining']:,} remaining")
print(f"executor cores: {spark.sparkContext.defaultParallelism}")

## 2 · Size on the executors, in batches, within the time budget

Progress is printed after each batch, and the Spark UI shows the running job as *SEACO-6161 sizing …*.
If the budget runs out, or you cancel, everything already sized is kept. Copy the run id from the summary above
into `resume_run_id` and run again to carry on.

In [0]:
spark.conf.set("seaco6161.run_id", RUN_ID)
spark.conf.set("seaco6161.run_ts", RUN_TS.strftime("%Y-%m-%d %H:%M:%S.%f"))
spark.conf.set("seaco6161.workspace", WORKSPACE_NAME)
spark.conf.set("seaco6161.audit_table", audit_table("dbfs_scan"))
spark.conf.set("seaco6161.batch_size", str(BATCH_SIZE))
spark.conf.set("seaco6161.budget_min", str(TIME_BUDGET_MIN))

In [0]:
%scala
scanInBatches("_scan_remaining", spark.conf.get("seaco6161.audit_table"),
              spark.conf.get("seaco6161.batch_size").toInt, spark.conf.get("seaco6161.budget_min").toDouble)

## 3 · Results

Sizes roll up from the leaves to every parent level. Directories not sized yet are left out, and the coverage
line says how complete the picture is.

In [0]:
scan = spark.table(audit_table("dbfs_scan")).filter(F.col("run_id") == RUN_ID) \
    .dropDuplicates(["path"])  # a batch cut short by a cancel may be re-sized on resume
parts = F.split(F.col("path"), "/")
rollup = None
for level in range(1, SCAN_DEPTH + 1):
    lvl = (scan.filter(F.size(parts) > level)
           .withColumn("prefix", F.concat_ws("/", F.slice(parts, 1, level + 1)))
           .groupBy("storage_class", "prefix").agg(F.sum("bytes").alias("bytes"), F.sum("files").alias("files"))
           .withColumn("level", F.lit(level)))
    rollup = lvl if rollup is None else rollup.unionByName(lvl)
rollup = rollup.withColumn("tib", F.round(F.col("bytes") / TIB, 3)).cache()
rollup.createOrReplaceTempView("dbfs_rollup")

sized = scan.filter("kind = 'dir'").count()
SCAN_STATS = dict(
    root_bytes=scan.filter("storage_class = 'root'").agg(F.sum("bytes")).collect()[0][0] or 0,
    errors=scan.filter("error IS NOT NULL").count(),
    coverage=sized / max(PLAN_COUNTS["planned"], 1),
)
display(rollup.filter("level = 1 AND storage_class = 'root'").orderBy(F.desc("bytes")))

In [0]:
print(f"Visible on the root account via dbfs:/ : {human(SCAN_STATS['root_bytes'])} "
      f"({SCAN_STATS['root_bytes'] / TIB / AZURE_REPORTED_TIB:.0%} of the {AZURE_REPORTED_TIB} TiB Azure reports)")
print(f"Coverage: {SCAN_STATS['coverage']:.1%} of planned directories sized; errors: {SCAN_STATS['errors']}")
if SCAN_STATS["coverage"] < 0.999:
    print(f"⚠️  Not complete. Re-run with resume_run_id = {RUN_ID}")

In [0]:
%sql
-- Every directory of 100 GiB or more at any level, biggest first
SELECT level, prefix, storage_class, tib, files
FROM dbfs_rollup
WHERE bytes >= 100 * power(1024, 3)
ORDER BY bytes DESC

In [0]:
# Slowest directories to size: usually never-vacuumed tables or huge log/tmp trees. Good cleanup leads.
display(scan.filter("kind = 'dir'")
        .select("path", F.round(F.col("bytes") / GIB, 1).alias("gib"), "files",
                F.round(F.col("millis") / 60000, 1).alias("minutes"), "error")
        .orderBy(F.desc("millis")).limit(25))